# MTI UNPAM Academic Assistant

Notebook ini berisi implementasi awal chatbot informasi akademik Program Magister Teknik Informatika UNPAM berbasis Text Mining, TF-IDF, dan Cosine Similarity.

## 1. Business Understanding

Tujuan project adalah membuat prototype chatbot yang membantu calon mahasiswa memahami struktur kurikulum, mata kuliah, peminatan, tesis, dan prospek karier MTI UNPAM.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from utils.chatbot_utils import load_faq_data, build_tfidf_engine, get_chatbot_response, evaluate_questions

DATA_PATH = PROJECT_ROOT / 'data' / 'faq_mti_unpam.csv'
EVAL_PATH = PROJECT_ROOT / 'data' / 'evaluation_questions.csv'

print('Project root:', PROJECT_ROOT)
print('Dataset path:', DATA_PATH)

## 2. Data Understanding

In [ ]:
faq_df = load_faq_data(DATA_PATH)
faq_df.head()

In [ ]:
print('Jumlah FAQ:', len(faq_df))
print('Jumlah kategori:', faq_df['category'].nunique())
faq_df.info()

## 3. Visualisasi Distribusi Dataset

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=faq_df, y='category', order=faq_df['category'].value_counts().index)
plt.title('Distribusi FAQ berdasarkan Kategori')
plt.xlabel('Jumlah FAQ')
plt.ylabel('Kategori')
plt.tight_layout()
plt.show()

In [ ]:
semester_df = faq_df.copy()
semester_df['semester'] = semester_df['semester'].replace('', 'Umum')
plt.figure(figsize=(10, 5))
sns.countplot(data=semester_df, x='semester', order=semester_df['semester'].value_counts().index)
plt.title('Distribusi FAQ berdasarkan Semester')
plt.xlabel('Semester')
plt.ylabel('Jumlah FAQ')
plt.tight_layout()
plt.show()

## 4. Modeling: TF-IDF dan Cosine Similarity

In [ ]:
vectorizer, tfidf_matrix = build_tfidf_engine(faq_df)
print('TF-IDF matrix shape:', tfidf_matrix.shape)

## 5. Pengujian Chatbot

In [ ]:
sample_questions = [
    'Apa saja mata kuliah semester 1?',
    'Saya tertarik cyber security cocok peminatan apa?',
    'Peminatan Data Science belajar apa?',
    'Apa resep membuat nasi goreng?'
]

for q in sample_questions:
    result = get_chatbot_response(q, faq_df, vectorizer, tfidf_matrix, threshold=0.25)
    print('='*90)
    print('Pertanyaan:', q)
    print('Kategori:', result['category'])
    print('Similarity:', result['similarity_score'])
    print('Jawaban:', result['answer'])

## 6. Evaluasi Sederhana

In [ ]:
eval_df = pd.read_csv(EVAL_PATH)
eval_result = evaluate_questions(eval_df, faq_df, vectorizer, tfidf_matrix, threshold=0.25)
eval_result

In [ ]:
accuracy = eval_result['is_correct'].mean()
print(f'Akurasi evaluasi sederhana: {accuracy:.2%}')

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=eval_result, x='similarity_score', y='test_question', hue='status')
plt.title('Similarity Score pada Pertanyaan Uji')
plt.xlabel('Similarity Score')
plt.ylabel('Pertanyaan Uji')
plt.tight_layout()
plt.show()

## 7. Kesimpulan

Prototype chatbot berhasil dibuat menggunakan pendekatan FAQ-based chatbot dengan TF-IDF dan Cosine Similarity. Sistem mampu menjawab pertanyaan yang sesuai dengan dataset FAQ dan memberikan fallback response untuk pertanyaan di luar konteks.